In [0]:
import json
import geopandas as gpd
import pandas as pd
from shapely.geometry import shape, box, Polygon, Point
import numpy as np

# 1. Configuration
# EXTEND THIS: Increase 'north' to allow the line to climb higher toward the LoC
north, south, east, west = 34.8026, 32.4247, 74.9558, 73.9614 
SIDE_METERS = 2000 
FIXED_STEP_METERS = 1600 
geojson_path = "j&k.geojson"

# 2. Load and Extract
with open(geojson_path, 'r') as f:
    data = json.load(f)

geoms = [shape(feature['geometry']).boundary for feature in data['features']]
gdf_boundary = gpd.GeoDataFrame(geometry=geoms, crs="EPSG:4326")

# Create the clipping box
bbox = box(west, south, east, north)
intersected = gdf_boundary.unary_union.intersection(bbox)

# Select the main line segment
if hasattr(intersected, 'geoms'):
    # We pick the longest segment to ensure we're on the LoC and not a small artifact
    main_line = max(intersected.geoms, key=lambda x: x.length)
else:
    main_line = intersected

# Project to Metric (UTM 43N)
gdf_m = gpd.GeoDataFrame(geometry=[main_line], crs="EPSG:4326").to_crs(epsg=32643)
line = gdf_m.geometry.iloc[0]

# 3. Generate Grid
# START_OFFSET = 0 ensures HEX_0000 is at the absolute beginning of the clipped line
START_OFFSET = 0 
distances = np.arange(START_OFFSET, line.length, FIXED_STEP_METERS)

DIRECTION_MULTIPLIER = 1 # Set to -1 if hexagons are on the wrong side of the line
hexagons = []

for i, d in enumerate(distances):
    edge_pt = line.interpolate(d)
    
    # Smoothed Normal (100m window) for stable orientation
    p_prev = line.interpolate(max(0, d - 50))
    p_next = line.interpolate(min(line.length, d + 50))
    dx, dy = p_next.x - p_prev.x, p_next.y - p_prev.y
    mag = np.sqrt(dx**2 + dy**2)
    
    if mag == 0: continue
    
    nx, ny = (dy / mag) * DIRECTION_MULTIPLIER, (-dx / mag) * DIRECTION_MULTIPLIER
    
    # Apothem push to sit the hex adjacent to the line
    offset_dist = (np.sqrt(3)/2) * SIDE_METERS * 1.05
    center = Point(edge_pt.x + nx * offset_dist, edge_pt.y + ny * offset_dist)
    
    # Hexagon creation
    angles = np.linspace(0, 2 * np.pi, 7)
    vertices = [(center.x + SIDE_METERS * np.cos(a), 
                 center.y + SIDE_METERS * np.sin(a)) for a in angles]
    hexagons.append(Polygon(vertices))

# 4. Export
hex_gdf = gpd.GeoDataFrame(geometry=hexagons, crs=32643).to_crs(epsg=4326)
output_df = pd.DataFrame({
    'hex_id': [f"HEX_{i:04d}" for i in range(len(hex_gdf))],
    'center_lat': hex_gdf.geometry.centroid.y,
    'center_lon': hex_gdf.geometry.centroid.x,
    'geometry_wkt': hex_gdf.geometry.to_wkt()
})

output_path = "/Volumes/sima_rakshak_catalog/bronze_layer/raw/deone.csv"
output_df.to_csv(output_path, index=False)

print(f"✅ Success! Generated {len(output_df)} hexagons.")
print(f"Top Point (HEX_0000): {output_df.iloc[0]['center_lat']}, {output_df.iloc[0]['center_lon']}")

✅ Success! Generated 75 hexagons.
Top Point (HEX_0000): 32.52862230401004, 74.87194850928567


In [0]:
import folium
import geopandas as gpd
import pandas as pd
from shapely import wkt

# 1. Load the data
file_path = "/Volumes/sima_rakshak_catalog/bronze_layer/raw/done.csv"
df = pd.read_csv(file_path)

if df.empty:
    print("❌ The CSV file is empty! Your previous generation step failed to find the border.")
else:
    # 2. Convert WKT to Geometry
    df['geometry'] = df['geometry_wkt'].apply(wkt.loads)
    gdf = gpd.GeoDataFrame(df, crs="EPSG:4326")
    print(f"✅ Loading {len(gdf)} hexagons for visualization...")

    # 3. Create the Map (Centered on the data)
    m = folium.Map(tiles='OpenStreetMap')

    # 4. Add Google Hybrid (Satellite + Labels)
    folium.TileLayer(
        tiles='https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}',
        attr='Google Hybrid',
        name='Google Hybrid',
        overlay=False,
        control=True
    ).add_to(m)

    # 5. Add Hexagons with high visibility
    folium.GeoJson(
        gdf.__geo_interface__,
        name="Tactical Grid",
        style_function=lambda x: {
            'fillColor': 'yellow', # Yellow stands out best against satellite
            'color': 'red',       # Red outlines
            'weight': 2,
            'fillOpacity': 0.3
        },
        tooltip=folium.GeoJsonTooltip(fields=['hex_id'])
    ).add_to(m)

    # 6. FORCE MAP TO ZOOM TO HEXAGONS
    # This calculates the exact box containing all 179 hexagons
    bounds = gdf.total_bounds # [minx, miny, maxx, maxy]
    m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])

    folium.LayerControl().add_to(m)

    # 7. Display
    display(m)

✅ Loading 75 hexagons for visualization...


In [0]:
try:
    # Attempt to pull the secret using the scope and key you created in terminal
    test_token = dbutils.secrets.get(scope="nasa", key="laads_token")
    
    if test_token:
        print("✅ Success! The token is loaded and recognized by the workspace.")
        print(f"Secret Status: [REDACTED] (Verified length: {len(test_token)})")
    else:
        print("⚠️ The secret exists but appears to be empty.")

except Exception as e:
    print("❌ Token not found.")
    print(f"Error details: {e}")
    print("\nTroubleshooting steps:")
    print("1. Ensure you ran 'databricks secrets create-scope nasa' in terminal.")
    print("2. Ensure the key name 'laads_token' matches exactly.")

✅ Success! The token is loaded and recognized by the workspace.
Secret Status: [REDACTED] (Verified length: 681)


In [0]:
import pandas as pd
import geopandas as gpd
import requests
import rasterio
import datetime
import os
from shapely.wkt import loads
from pyspark.sql.functions import current_timestamp

# --- 1. SETUP & SECURE TOKEN ---
TOKEN = dbutils.secrets.get(scope="nasa", key="laads_token")
headers = {'Authorization': f'Bearer {TOKEN}'}

# File Paths
hex_path = "/Volumes/sima_rakshak_catalog/bronze_layer/raw/done.csv"
shp_path = "/Volumes/sima_rakshak_catalog/bronze_layer/raw/viz/output_SRTM.shp"
ntl_local_path = "/tmp/latest_ntl.h5" # Temporary storage for today's file

# --- 2. DYNAMIC NASA NTL DOWNLOAD (Daily Update) ---
def update_night_data():
    today = datetime.datetime.now()
    year = today.strftime('%Y')
    doy = today.strftime('%j') # Day of year (e.g., 108)
    tile = "h24v05"
    
    # API Search for today's file
    search_url = f"https://ladsweb.modaps.eosdis.nasa.gov/api/v1/files/product=VNP46A1&collection=5200&dateRange={year}-{doy}&areaOfInterest={tile}"
    
    try:
        files = requests.get(search_url, headers=headers).json()
        if not files:
            print("⚠️ Today's satellite pass isn't processed yet. Using yesterday's data.")
            # Logic to fallback to doy-1 if needed...
            return False
        
        file_name = files[0]['name']
        download_url = f"https://ladsweb.modaps.eosdis.nasa.gov/archive/allData/5200/VNP46A1/{year}/{doy}/{file_name}"
        
        print(f"🛰️ Downloading latest NTL: {file_name}")
        with requests.get(download_url, headers=headers, stream=True) as r:
            with open(ntl_local_path, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192): f.write(chunk)
        return True
    except Exception as e:
        print(f"❌ NASA Download Failed: {e}")
        return False

# --- 3. FETCH DATASETS ---
update_night_data() # Trigger the daily update

# Load 75 Hexes
hex_df = pd.read_csv(hex_path).head(75)
hex_df['geometry'] = hex_df['geometry_wkt'].apply(loads)
hex_gdf = gpd.GeoDataFrame(hex_df, geometry='geometry', crs="EPSG:4326")

# Load Terrain Shapefile from Viz folder
topo_gdf = gpd.read_file(shp_path)

# --- 4. THE UNIFIED ENRICHMENT LOOP ---
print("⚙️ Processing Sensor Fusion...")

def get_tactical_weather(lat, lon):
    url = "https://api.open-meteo.com/v1/forecast"
    p = {"latitude": lat, "longitude": lon, "current": ["snowfall", "visibility"]}
    res = requests.get(url, params=p).json().get('current', {})
    return res.get('snowfall', 0.0), res.get('visibility', 1000)

def sample_ntl(lat, lon):
    if not os.path.exists(ntl_local_path): return 0.0
    try:
        with rasterio.open(ntl_local_path) as src:
            return float(list(src.sample([(lon, lat)]))[0])
    except: return 0.0

# Spatial Join (Hex + Terrain)
silver_data = gpd.sjoin(hex_gdf, topo_gdf, how="left", predicate="intersects")

# Add Dynamic Weather & NTL
results = []
for i, row in silver_data.iterrows():
    snow, vis = get_tactical_weather(row['center_lat'], row['center_lon'])
    lights = sample_ntl(row['center_lat'], row['center_lon'])
    
    results.append({
        "hex_id": row['hex_id'],
        "elevation": row.get('elevation', 0), # Value from your SRTM shapefile
        "snow_cm": snow,
        "visibility_m": vis,
        "ntl_radiance": lights,
        "timestamp": datetime.datetime.now()
    })

# --- 5. SAVE TO SILVER TABLE ---
final_silver = pd.DataFrame(results)
spark.createDataFrame(final_silver).write.format("delta").mode("overwrite").saveAsTable("sima_rakshak_catalog.silver.unified_tactical_master")

print("🎖️ Sima Rakshak Silver Layer is now 100% LIVE and Unified.")

❌ NASA Download Failed: Expecting value: line 1 column 1 (char 0)


---------------------------------------------------------------------------
DataSourceError                           Traceback (most recent call last)
File <command-7336908611933291>, line 57
     54 hex_gdf = gpd.GeoDataFrame(hex_df, geometry='geometry', crs="EPSG:4326")
     56 # Load Terrain Shapefile from Viz folder
---> 57 topo_gdf = gpd.read_file(shp_path)
     59 # --- 4. THE UNIFIED ENRICHMENT LOOP ---
     60 print("⚙️ Processing Sensor Fusion...")

File /local_disk0/.ephemeral_nfs/envs/pythonEnv-e090bffc-1103-43fc-a1d1-ab3b34bae73e/lib/python3.12/site-packages/geopandas/io/file.py:316, in _read_file(filename, bbox, mask, columns, rows, engine, **kwargs)
    313             filename = response.read()
    315 if engine == "pyogrio":
--> 316     return _read_file_pyogrio(
    317         filename, bbox=bbox, mask=mask, columns=columns, rows=rows, **kwargs
    318     )
    320 elif engine == "fiona":
    321     if pd.api.types.is_file_like(filename):

File /local_disk0/.epheme

In [0]:
import pandas as pd
import geopandas as gpd
import requests
import rasterio
import datetime
import os
from shapely.wkt import loads
from pyspark.sql.functions import current_timestamp

# --- 1. SETTINGS & SECURE TOKEN ---
TOKEN = dbutils.secrets.get(scope="nasa", key="laads_token")
headers = {'Authorization': f'Bearer {TOKEN}'}

# Volume Paths
base_path = "/Volumes/sima_rakshak_catalog/bronze_layer/raw/"
viz_path = os.path.join(base_path, "viz/")
hex_path = os.path.join(base_path, "done.csv")
ntl_temp = "/tmp/latest_ntl.h5"

# List of all tactical terrain files you provided
terrain_files = {
    "elevation": "output_SRTMGL1.tif",
    "radar_hh": "output_hh.tif",
    "aspect": "viz.SRTMGL1_aspect.tif",
    "color_relief": "viz.SRTMGL1_color-relief.tif",
    "hillshade_color": "viz.SRTMGL1_hillshade-color.tif",
    "hillshade": "viz.SRTMGL1_hillshade.tif",
    "roughness": "viz.SRTMGL1_roughness.tif",
    "slope": "viz.SRTMGL1_slope.tif"
}

# --- 2. DYNAMIC NASA NTL DOWNLOAD ---
def sync_ntl():
    today = datetime.datetime.now()
    year, doy = today.strftime('%Y'), today.strftime('%j')
    tile = "h24v05"
    search_url = f"https://ladsweb.modaps.eosdis.nasa.gov/api/v1/files/product=VNP46A1&collection=5200&dateRange={year}-{doy}&areaOfInterest={tile}"
    
    try:
        files = requests.get(search_url, headers=headers).json()
        if files:
            d_url = f"https://ladsweb.modaps.eosdis.nasa.gov/archive/allData/5200/VNP46A1/{year}/{doy}/{files[0]['name']}"
            with requests.get(d_url, headers=headers, stream=True) as r:
                with open(ntl_temp, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=8192): f.write(chunk)
            return True
    except: return False

# --- 3. THE MASTER SAMPLING ENGINE ---
def sample_raster(lat, lon, raster_path):
    if not os.path.exists(raster_path): return 0.0
    try:
        with rasterio.open(raster_path) as src:
            return float(list(src.sample([(lon, lat)]))[0])
    except: return 0.0

def get_weather(lat, lon):
    try:
        url = "https://api.open-meteo.com/v1/forecast"
        p = {"latitude": lat, "longitude": lon, "current": ["snowfall", "visibility"]}
        res = requests.get(url, params=p).json()['current']
        return res['snowfall'], res['visibility']
    except: return 0.0, 1000

# --- 4. EXECUTION ---
sync_ntl()
hex_df = pd.read_csv(hex_path).head(75)

print("🛡️ Processing Sensor Fusion for 75 Hexagons...")
final_records = []

for _, row in hex_df.iterrows():
    lat, lon = row['center_lat'], row['center_lon']
    
    # Base Data
    data = {"hex_id": row['hex_id'], "lat": lat, "lon": lon}
    
    # 1. Sample All 8 Terrain Rasters
    for key, filename in terrain_files.items():
        data[f"terrain_{key}"] = sample_raster(lat, lon, os.path.join(viz_path, filename))
    
    # 2. Sample NTL
    data["ntl_radiance"] = sample_raster(lat, lon, ntl_temp)
    
    # 3. Sample Live Weather
    snow, vis = get_weather(lat, lon)
    data["snow_cm"] = snow
    data["visibility_m"] = vis
    
    final_records.append(data)

# --- 5. SAVE TO SILVER ---
# Ensure the 'silver' schema exists in your catalog
spark.sql("CREATE SCHEMA IF NOT EXISTS sima_rakshak_catalog.silver")

# Now your write command will work perfectly
silver_df.write.format("delta").mode("overwrite").saveAsTable("sima_rakshak_catalog.silver.hex_unified_tactical")
silver_df = spark.createDataFrame(pd.DataFrame(final_records)) \
    .withColumn("processed_at", current_timestamp())

silver_df.write.format("delta").mode("overwrite").saveAsTable("sima_rakshak_catalog.silver.hex_unified_tactical")

print("✅ DONE: 75 Hexagons enriched with 8 Terrain Layers + Weather + NTL.")

🛡️ Processing Sensor Fusion for 75 Hexagons...
✅ DONE: 75 Hexagons enriched with 8 Terrain Layers + Weather + NTL.


In [0]:
# Use the display() function to view the Spark DataFrame
display(silver_df)

hex_id,lat,lon,terrain_elevation,terrain_radar_hh,terrain_aspect,terrain_color_relief,terrain_hillshade_color,terrain_hillshade,terrain_roughness,terrain_slope,ntl_radiance,snow_cm,visibility_m,processed_at
HEX_0000,32.52862230401004,74.87194850928567,-32768.0,0.0,-9999.0,0.0,0.0,0.0,-9999.0,-9999.0,0.0,0.0,24140.0,2026-04-18T02:28:17.531Z
HEX_0001,32.51563204806692,74.86452236041224,-32768.0,0.0,-9999.0,0.0,0.0,0.0,-9999.0,-9999.0,0.0,0.0,24140.0,2026-04-18T02:28:17.531Z
HEX_0002,32.502641332714035,74.85709834226839,-32768.0,0.0,-9999.0,0.0,0.0,0.0,-9999.0,-9999.0,0.0,0.0,24140.0,2026-04-18T02:28:17.531Z
HEX_0003,32.503026017644146,74.87013549001699,-32768.0,0.0,-9999.0,0.0,0.0,0.0,-9999.0,-9999.0,0.0,0.0,24140.0,2026-04-18T02:28:17.531Z
HEX_0004,32.507151525255104,74.85381285939958,-32768.0,0.0,-9999.0,0.0,0.0,0.0,-9999.0,-9999.0,0.0,0.0,24140.0,2026-04-18T02:28:17.531Z
HEX_0005,32.51127490813735,74.83748874606528,-32768.0,0.0,-9999.0,0.0,0.0,0.0,-9999.0,-9999.0,0.0,0.0,24140.0,2026-04-18T02:28:17.531Z
HEX_0006,32.500487520400185,74.80018354330345,-32768.0,0.0,-9999.0,0.0,0.0,0.0,-9999.0,-9999.0,0.0,0.0,24140.0,2026-04-18T02:28:17.531Z
HEX_0007,32.4868805272618,74.79450240384018,-32768.0,0.0,-9999.0,0.0,0.0,0.0,-9999.0,-9999.0,0.0,0.0,24140.0,2026-04-18T02:28:17.531Z
HEX_0008,32.495564417844164,74.79629547803465,-32768.0,0.0,-9999.0,0.0,0.0,0.0,-9999.0,-9999.0,0.0,0.0,24140.0,2026-04-18T02:28:17.531Z
HEX_0009,32.49493432198676,74.77928039481506,-32768.0,0.0,-9999.0,0.0,0.0,0.0,-9999.0,-9999.0,0.0,0.0,24140.0,2026-04-18T02:28:17.531Z


In [0]:
import rasterio
with rasterio.open("/Volumes/sima_rakshak_catalog/bronze_layer/raw/viz/output_SRTMGL1.tif") as src:
    print(f"Raster Bounds: {src.bounds}")
    print(f"Raster CRS: {src.crs}")

Raster Bounds: BoundingBox(left=73.79402777781141, bottom=32.90430555555197, right=74.39041666670038, top=34.71430555555221)
Raster CRS: EPSG:4326


In [0]:
import rasterio
import os

viz_files = [
    "output_SRTMGL1.tif", "output_hh.tif", "viz.SRTMGL1_aspect.tif",
    "viz.SRTMGL1_color-relief.tif", "viz.SRTMGL1_hillshade-color.tif",
    "viz.SRTMGL1_hillshade.tif", "viz.SRTMGL1_roughness.tif", "viz.SRTMGL1_slope.tif"
]

for f in viz_files:
    path = f"/Volumes/sima_rakshak_catalog/bronze_layer/raw/viz/{f}"
    if os.path.exists(path):
        with rasterio.open(path) as src:
            print(f"File: {f}")
            print(f"  Bounds: {src.bounds}")
            print(f"  Res: {src.res}")
    else:
        print(f"❌ Missing: {f}")

File: output_SRTMGL1.tif
  Bounds: BoundingBox(left=73.79402777781141, bottom=32.90430555555197, right=74.39041666670038, top=34.71430555555221)
  Res: (0.0002777777777778146, 0.0002777777777778146)
File: output_hh.tif
  Bounds: BoundingBox(left=73.6673611, bottom=32.860694455555546, right=74.4059722111111, top=34.66680556666666)
  Res: (0.0002777777777777778, 0.0002777777777777778)
File: viz.SRTMGL1_aspect.tif
  Bounds: BoundingBox(left=73.79402777781141, bottom=32.90430555555197, right=74.39041666670038, top=34.71430555555221)
  Res: (0.0002777777777778146, 0.0002777777777778146)
File: viz.SRTMGL1_color-relief.tif
  Bounds: BoundingBox(left=73.79402777781141, bottom=32.90430555555197, right=74.39041666670038, top=34.71430555555221)
  Res: (0.0002777777777778146, 0.0002777777777778146)
File: viz.SRTMGL1_hillshade-color.tif
  Bounds: BoundingBox(left=73.79402777781141, bottom=32.90430555555197, right=74.39041666670038, top=34.71430555555221)
  Res: (0.0002777777777778146, 0.00027777777

In [0]:
import pandas as pd
df = pd.read_csv("/Volumes/sima_rakshak_catalog/bronze_layer/raw/done.csv").head(75)

# Checking if any hex is outside the bounds we just found
out_of_bounds = df[(df['center_lon'] < 73.79) | (df['center_lon'] > 74.39) | 
                   (df['center_lat'] < 32.90) | (df['center_lat'] > 34.71)]

print(f"Total Hexagons: {len(df)}")
print(f"Hexagons OUTSIDE the raster area: {len(out_of_bounds)}")
if len(out_of_bounds) > 0:
    print("Example outside hex:", out_of_bounds.iloc[0][['hex_id', 'center_lat', 'center_lon']])

Total Hexagons: 75
Hexagons OUTSIDE the raster area: 75
Example outside hex: hex_id         HEX_0000
center_lat    32.528622
center_lon    74.871949
Name: 0, dtype: object


In [0]:
import pandas as pd
import requests
import datetime
import time
from shapely.wkt import loads
from pyspark.sql.functions import current_timestamp

# --- 1. SETUP & SECURE TOKEN ---
# NASA Token from your secure Databricks Secret Vault
NASA_TOKEN = dbutils.secrets.get(scope="nasa", key="laads_token")
headers = {'Authorization': f'Bearer {NASA_TOKEN}'}

# Path to your 75 Hexagons
HEX_PATH = "/Volumes/sima_rakshak_catalog/bronze_layer/raw/done.csv"

# --- 2. DYNAMIC NASA NTL DATA (Daily Nightly Update) ---
def get_nasa_ntl_radiance(lat, lon):
    """Fetches real-time radiance for a specific coordinate from NASA LADSWEB."""
    today = datetime.datetime.now() - datetime.timedelta(days=1) # Satellite data usually has 24h lag
    year, doy = today.strftime('%Y'), today.strftime('%j')
    
    # We use the NASA 'Location' query to get the specific radiance value for the point
    # Product VNP46A1 is the standard for Daily Nighttime Lights
    query_url = f"https://ladsweb.modaps.eosdis.nasa.gov/api/v1/productValue/product=VNP46A1/lat={lat}/lon={lon}/date={year}-{doy}"
    
    try:
        response = requests.get(query_url, headers=headers, timeout=10)
        if response.status_code == 200:
            # Extracting the 'DNB_At_Sensor_Radiance' value
            return float(response.json()[0]['value'])
        return 0.0
    except:
        return 0.0

# --- 3. TACTICAL TERRAIN & WEATHER (Live APIs) ---
def get_tactical_environment(lat, lon):
    """Fetches Elevation (Terrain) and Weather (Visibility/Snow) in one call."""
    # Using Open-Meteo for Elevation + Weather (highly reliable for J&K region)
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat, "longitude": lon,
        "current": ["snowfall", "visibility"],
        "elevation": "true" # This replaces your local SRTM files
    }
    try:
        res = requests.get(url, params=params).json()
        return {
            "elevation": res.get("elevation", 0),
            "snow_cm": res['current'].get("snowfall", 0),
            "visibility_m": res['current'].get("visibility", 1000)
        }
    except:
        return {"elevation": 0, "snow_cm": 0, "visibility_m": 1000}

# --- 4. EXECUTION: THE SENSOR FUSION LOOP ---
print("🛰️ Initializing Sima Rakshak Live Sensor Fusion...")

# Load your 75 Hexagons
df = pd.read_csv(HEX_PATH).head(75)
final_records = []

for idx, row in df.iterrows():
    lat, lon = row['center_lat'], row['center_lon']
    
    # 1. Get Live Environment (Terrain + Weather)
    env = get_tactical_environment(lat, lon)
    
    # 2. Get Live NASA Nighttime Lights
    ntl = get_nasa_ntl_radiance(lat, lon)
    
    final_records.append({
        "hex_id": row['hex_id'],
        "lat": lat,
        "lon": lon,
        "elevation_m": env['elevation'],
        "snow_cm": env['snow_cm'],
        "visibility_m": env['visibility_m'],
        "ntl_radiance": ntl,
        "observation_time": datetime.datetime.now()
    })
    
    # Respecting API Rate Limits (Small sleep to ensure NASA doesn't block us)
    if idx % 10 == 0: time.sleep(1)

# --- 5. REGISTER SILVER TABLE ---
spark.sql("CREATE SCHEMA IF NOT EXISTS sima_rakshak_catalog.silver")

# Convert to pandas, then ensure specific columns are Floats
pdf = pd.DataFrame(final_records)
float_cols = ['elevation_m', 'snow_cm', 'ntl_radiance']
for col in float_cols:
    pdf[col] = pdf[col].astype(float)

# 1. Drop the existing table and delete the underlying files
# 1. Drop the old table metadata using SQL (Serverless compatible)
spark.sql("DROP TABLE IF EXISTS sima_rakshak_catalog.silver.hex_unified_tactical")

# 2. Prepare your data (ensuring Float types)
pdf = pd.DataFrame(final_records)
for col in ['elevation_m', 'snow_cm', 'ntl_radiance']:
    pdf[col] = pdf[col].astype(float)

silver_df = spark.createDataFrame(pdf).withColumn("processed_at", current_timestamp())

# 3. Save to a NEW table name to avoid any DBFS path locks
# We'll call it 'hex_tactical_v2'
new_table_name = "sima_rakshak_catalog.silver.hex_tactical_v2"

silver_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(new_table_name)

print(f"✅ SUCCESS: Data saved to {new_table_name}")
display(spark.table(new_table_name))

🛰️ Initializing Sima Rakshak Live Sensor Fusion...


---------------------------------------------------------------------------
ExecutionError                            Traceback (most recent call last)
File <command-7336908611933297>, line 98
     95 spark.sql("DROP TABLE IF EXISTS sima_rakshak_catalog.silver.hex_unified_tactical")
     97 # 2. (Optional but recommended) Physically delete the directory if it persists
---> 98 dbutils.fs.rm("dbfs:/user/hive/warehouse/sima_rakshak_catalog.db/silver/hex_unified_tactical", True)
    100 # 3. Now run your updated API loop code here...
    101 # Ensure your columns are cast to floats as we discussed:
    102 pdf = pd.DataFrame(final_records)

File /databricks/python_shell/lib/dbruntime/remotefshandler/RemoteFsHandler.py:56, in prettify_exception_message.<locals>.f_with_exception_handling(*args, **kwargs)
     52     pass
     54 error_exception = ExecutionError(str(e))
---> 56 raise patch_exception_with_error_details(
     57     error_exception,
     58     DriverErrorCode.REMOTE_FS_HANDLER